# ESM3/C

In [5]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
# only load this one time per session
if 'NOTEBOOK_INITIALIZED' not in globals():
    os.chdir(os.path.dirname(os.path.abspath('.')))
    NOTEBOOK_INITIALIZED = True

import src.utils as utils
import src.config as config
import src.haplosaurus as hs

# import src.vep_pipeline as vp
import src.vep_metrics as vm

import src.ESM3 as ESM3

## ESM3

In [2]:
import torch
available_gpus = [torch.cuda.device(i) for i in range(torch.cuda.device_count())]
available_gpus

In [3]:
devices = vm.get_available_gpus()
devices

[(0, 84974239744), (1, 84974239744), (2, 84974239744), (3, 84974239744)]

In [4]:
import torch
devices = []
for i in range(torch.cuda.device_count()):
   devices.append(torch.cuda.get_device_properties(i))
devices

[_CudaDeviceProperties(name='NVIDIA A100 80GB PCIe', major=8, minor=0, total_memory=81037MB, multi_processor_count=108, uuid=57a5dbce-e7ee-766b-1803-e35688317a0c, L2_cache_size=40MB),
 _CudaDeviceProperties(name='NVIDIA A100 80GB PCIe', major=8, minor=0, total_memory=81037MB, multi_processor_count=108, uuid=ee601dc3-6f2c-3e1b-8b01-5c4012c77075, L2_cache_size=40MB),
 _CudaDeviceProperties(name='NVIDIA A100 80GB PCIe', major=8, minor=0, total_memory=81037MB, multi_processor_count=108, uuid=d1d17271-1e44-abb0-9cf6-de92acf10943, L2_cache_size=40MB),
 _CudaDeviceProperties(name='NVIDIA A100 80GB PCIe', major=8, minor=0, total_memory=81037MB, multi_processor_count=108, uuid=ca14fd86-8f12-0e67-15c5-eefe075aa2dd, L2_cache_size=40MB)]

In [2]:
import torch
torch.cuda.set_device(1)

In [6]:
from huggingface_hub import login   
from esm.models.esm3 import ESM3
from esm.sdk.api import ESM3InferenceClient, ESMProtein, GenerationConfig


# Will instruct you how to get an API key from huggingface hub, make one with "Read" permission.
# login( token="{HUGGINGFACE_TOKEN}")

# This will download the model weights and instantiate the model on your machine.
model: ESM3InferenceClient = ESM3.from_pretrained("esm3-open").to("cuda")  # Use GPU 2 which has the most available memory

# # Generate a completion for a partial Carbonic Anhydrase (2vvb)
prompt = "___________________________________________________DQATSLRILNNGHAFNVEFDDSQDKAVLKGGPLDGTYRLIQFHFHWGSLDGQGSEHTVDKKKYAAELHLVHWNTKYGDFGKAVQQPDGLAVLGIFLKVGSAKPGLQKVVDVLDSIKTKGKSADFTNFDPRGLLPESLDYWTYPGSLTTPP___________________________________________________________"
protein = ESMProtein(sequence=prompt)

# Generate the sequence, then the structure. This will iteratively unmask the sequence track.
protein = model.generate(protein, GenerationConfig(track="sequence", num_steps=8, temperature=0.7))

# We can show the predicted structure for the generated sequence.
protein = model.generate(protein, GenerationConfig(track="structure", num_steps=8))
protein.to_pdb("./generation.pdb")

# Then we can do a round trip design by inverse folding the sequence and recomputing the structure
protein.sequence = None
protein = model.generate(protein, GenerationConfig(track="sequence", num_steps=8))
protein.coordinates = None
protein = model.generate(protein, GenerationConfig(track="structure", num_steps=8))
protein.to_pdb("./round_tripped.pdb")

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:00<00:00, 11.69it/s]
/home/schilder/.conda/envs/esm3/lib/python3.12/site-packages/esm/utils/structure/protein_complex.py:223: UserWarning: Entity ID not found in metadata, using None as default
  warnings.warn("Entity ID not found in metadata, using None as default")
/home/schilder/.conda/envs/esm3/lib/python3.12/site-packages/esm/models/vqvae.py:286: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=False):  # type: ignore
100%|██████████| 8/8 [00:00<00:00, 11.70it/s]


In [7]:
protein

ESMProtein(sequence='SASEKEWNYDPEKWCEEGFETCCEGKNQSPINIVTDKAVYNSSLTPLKINYNPELVKEIVNTGHTAQVNFDMSKKKNTLTGGPLDGEYILKQFHLHWGSTNDKGSEHTIDGKSYAAELHLVHWNEKYGSFEEAVKKPDGLAVLGIFLEVGKANEGFQKICDVLPKIKYKGQSTELKNFDPNGLLPNDLSYYTYNGSLTTPPCYESVIWTVFKTPITISQEQLDKFRELQLNEGDKKVPLVDNFRPVQPLNGRTVYYHKLK', secondary_structure=None, sasa=None, function_annotations=None, coordinates=tensor([[[-21.0247,  27.2188,   1.4824],
         [-20.3265,  26.0066,   1.0672],
         [-19.2007,  25.6618,   2.0365],
         ...,
         [     inf,      inf,      inf],
         [     inf,      inf,      inf],
         [     inf,      inf,      inf]],

        [[-19.2725,  25.4507,   2.9718],
         [-18.1352,  25.1545,   3.8366],
         [-17.0126,  24.4792,   3.0559],
         ...,
         [     inf,      inf,      inf],
         [     inf,      inf,      inf],
         [     inf,      inf,      inf]],

        [[-16.2438,  24.8768,   3.0843],
         [-15.0285,  24.2981,   2.5210],
         [-14.6762,  22.9833,   3.

## ESMC

### Local

In [8]:
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

protein = ESMProtein(sequence="AAAAA")

client = ESMC.from_pretrained("esmc_300m").to("cuda") # or "cpu"
protein_tensor = client.encode(protein)
logits_output = client.logits(
   protein_tensor, LogitsConfig(sequence=True, return_embeddings=True)
)
print(logits_output.logits, logits_output.embeddings)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

ForwardTrackData(sequence=tensor([[[-38.2500, -38.0000, -38.0000,  12.5625,  21.6250,  22.2500,  22.0000,
           21.8750,  21.5000,  21.6250,  21.7500,  21.3750,  20.7500,  21.5000,
           22.0000,  21.0000,  20.7500,  20.3750,  20.3750,  20.2500,  21.8750,
           19.8750,  19.8750,  19.3750,  18.3750,   1.1875,  -1.7812,  -4.1250,
          -20.7500, -38.0000, -38.0000, -38.2500, -38.0000, -38.2500, -38.2500,
          -38.2500, -38.0000, -38.0000, -38.0000, -38.0000, -38.0000, -38.0000,
          -38.0000, -38.0000, -38.0000, -38.2500, -38.2500, -38.0000, -38.2500,
          -38.0000, -38.2500, -38.0000, -38.0000, -38.2500, -38.0000, -38.0000,
          -38.0000, -38.0000, -38.0000, -38.0000, -38.2500, -38.0000, -38.0000,
          -38.0000],
         [-40.0000, -40.0000, -40.0000,   5.4062,  20.1250,  19.8750,  18.7500,
           20.6250,  18.8750,  18.3750,  18.6250,  18.5000,  18.1250,  18.0000,
           18.7500,  17.7500,  17.3750,  17.1250,  17.7500,  16.8750,  22

In [ ]:
print(logits_output.logits.sequence.shape)
print(logits_output.embeddings.shape)

torch.Size([1, 7, 64])
torch.Size([1, 7, 960])


### ESM Forge

In [29]:
from getpass import getpass

token = getpass("Token from Forge console: ")

In [ ]:
# Model	Model Size	Number of Layers	Release Date
# esmc-6b-2024-12	6B	80	2024-12
# esmc-600m-2024-12	600M	36	2024-12
# esmc-300m-2024-12	300M	30	2024-12

model_name = "esmc_300m" #"esmc-300m-2024-12" #

## Remote
run_local = True
if not run_local:
    from esm.sdk import client
    model = client(
        model=model_name, 
        url="https://forge.evolutionaryscale.ai", 
        token=token
    )
## Local
else:
    import torch
    from esm.models.esmc import ESMC
    torch.cuda.set_device(1)
    model = ESMC.from_pretrained(model_name).to("cuda")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
from concurrent.futures import ThreadPoolExecutor
from typing import Sequence

from esm.sdk.api import (
    ESM3InferenceClient,
    ESMProtein,
    ESMProteinError,
    LogitsConfig,
    LogitsOutput,
    ProteinType,
)

EMBEDDING_CONFIG = LogitsConfig(
    sequence=True, 
    return_embeddings=True, 
    return_hidden_states=True
)


def embed_sequence(model: ESM3InferenceClient, sequence: str) -> LogitsOutput:
    protein = ESMProtein(sequence=sequence)
    protein_tensor = model.encode(protein)
    output = model.logits(protein_tensor, EMBEDDING_CONFIG)
    return output


def embed_sequence_parallel(
    model: ESM3InferenceClient, 
    inputs: Sequence[ProteinType],
    max_workers: int = 1,
    error: bool = False
) -> Sequence[LogitsOutput]:
    """Forge supports auto-batching. So batch_embed() is as simple as running a collection
    of embed calls in parallel using asyncio.
    """
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(embed_sequence, model, protein) for protein in inputs
        ]
        results = []
        for future in futures:
            try:
                results.append(future.result())
            except Exception as e:
                if error:
                    raise e
                else:
                    results.append(ESMProteinError(500, str(e)))
    return results

## Read in haplotypes

In [43]:
tx_ids = hs.list_haplotypes()

Found 47325 haplotypes in: '/home/schilder/.cache/ensembl_rest/haplotypes'


In [44]:
haplotypes = hs.get_haplotypes(cache_only=True, 
                               max_tx_ids=10)
hap_seqs = hs.get_haplotype_seqs(haplotypes, 
                                 aligned=0,
                                 add_haplotype_names=2,
                                 key='protein_haplotypes')

# %%


Found 47325 haplotypes in: '/home/schilder/.cache/ensembl_rest/haplotypes'


Getting haplotypes:   0%|          | 0/10 [00:00<?, ?it/s]

Adding reference haplotype:   0%|          | 0/10 [00:00<?, ?it/s]

Getting haplotype sequences:   0%|          | 0/10 [00:00<?, ?it/s]

Getting haplotype names:   0%|          | 0/10 [00:00<?, ?it/s]

In [45]:
hap_df = hs.haplotypes_to_df(haplotypes)
hap_df

Adding reference haplotype:   0%|          | 0/10 [00:00<?, ?it/s]

Getting haplotype sequences:   0%|          | 0/10 [00:00<?, ?it/s]

Getting haplotype names:   0%|          | 0/10 [00:00<?, ?it/s]

,ENST,ENSP,sequence
haplotype,,,
ENSP00000003084:REF,ENST00000269571,ENSP00000003084,MQRSPLEKASVVSKLFFSWTRPILRKGYRQRLELSDIYQIPSVDSA...
ENSP00000003084:470V>M,ENST00000269571,ENSP00000003084,MQRSPLEKASVVSKLFFSWTRPILRKGYRQRLELSDIYQIPSVDSA...
ENSP00000003084:556I>V,ENST00000269571,ENSP00000003084,MQRSPLEKASVVSKLFFSWTRPILRKGYRQRLELSDIYQIPSVDSA...
ENSP00000003084:75R>Q,ENST00000269571,ENSP00000003084,MQRSPLEKASVVSKLFFSWTRPILRKGYRQRLELSDIYQIPSVDSA...
"ENSP00000003084:74R>W,1270D>N",ENST00000269571,ENSP00000003084,MQRSPLEKASVVSKLFFSWTRPILRKGYRQRLELSDIYQIPSVDSA...
...,...,...,...
ENSP00000269571:1056G>S,ENST00000269571,ENSP00000269571,MELAALCRWGLLLALLPPGAASTQVCTGTDMKLRLPASPETHLDML...
"ENSP00000269571:1073S>C,1170P>A",ENST00000269571,ENSP00000269571,MELAALCRWGLLLALLPPGAASTQVCTGTDMKLRLPASPETHLDML...
ENSP00000269571:197P>L,ENST00000269571,ENSP00000269571,MELAALCRWGLLLALLPPGAASTQVCTGTDMKLRLPASPETHLDML...


In [ ]:
outputs = embed_sequence_parallel(model=model, 
                                  inputs=hap_df["sequence"].tolist())
outputs

## ESM++

https://huggingface.co/Synthyra/ESMplusplus_small

https://github.com/evolutionaryscale/esm/issues/176#issuecomment-2568427933


In [7]:
import torch
from transformers import AutoModelForMaskedLM #AutoModel also works

torch.cuda.set_device(1)

model = AutoModelForMaskedLM.from_pretrained('Synthyra/ESMplusplus_small',
                                              trust_remote_code=True)
model.to("cuda")
print()

In [ ]:
save_dir = "/home/schilder/projects/data/1000_Genomes_on_GRCh38/embeddings/ESMplusplus_small/"
ESM3.embed_sequences(model = model,
                      save_dir = save_dir, 
                      batch_size = 500)

### Get total size of embeddings

In [7]:
# Get total number of files in the directory
!ls -l {save_dir} | wc -l

9316


In [8]:
# Get total size of embeddings
!du -sh {save_dir}

3.0G	/home/schilder/projects/data/1000_Genomes_on_GRCh38/embeddings/ESMplusplus_small/
